# Fig. 3e–g — Probability matching: attractor geometry → behaviour

Same geometry→behaviour arc as the reward-magnitude panels (`main_paper/fig2/f6ii_mag.ipynb`).

**Headline (the open question, decided).** *Attractor body length sets within-block choice variability*: a long, soft attractor body lets the state diffuse along it under noise, so the choice ratio wanders a lot within a block; a short, point-like body pins the state, so choice is stable. Demonstrated here by lengthening PC's body with the weight-update rate $\beta$ (noise fixed): body length (branch L/W) and within-block choice SD rise together. This is the behavioural face of the F3_2 branch-length→choice-SD result.

**Mechanism underneath.** The attractor coordinate is the equilibrium manifold position $\Delta w=w_A-w_B$; choice ratio is a fixed sigmoid of it, and the reward *ratio* sets where the attractor equilibrates. A long body is exactly what lets $\Delta w$ wander.

**Self-contained:** all simulation + metric code lives in this notebook (next cell); the only external dependency is the base `ssb` package. Heavy runs are gated by `FULL`. Figures save as **SVG** (clouds/scatter rasterized).

In [ ]:
# ============================================================================
#  Setup + all simulation / metric helpers (inlined — only base `ssb` external)
# ============================================================================
import os
os.environ.setdefault('JAX_PLATFORM_NAME', 'cpu'); os.environ.setdefault('JAX_PLATFORMS', 'cpu')
import warnings; warnings.filterwarnings('ignore')
import sys
import numpy as np, time
import matplotlib.pyplot as plt, matplotlib as mpl
import matplotlib.ticker as mticker, matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter
mpl.rcParams['font.family'] = 'Arial'; mpl.rcParams['svg.fonttype'] = 'none'
import logging
for _n in ('matplotlib.font_manager','jax','absl','jax._src.xla_bridge'):
    logging.getLogger(_n).setLevel(logging.ERROR)

import jax, jax.numpy as jnp
from ssb.models import ssb, rules
from ssb.utils.batch_inits import make_default_hps, batchify_hps, init_params_batch
from ssb.envs.bandits import bandit_bait

# ── constants ────────────────────────────────────────────────────────────────
RULE_COL = {'PC': '#7B4377', 'COV': '#7FAF87', 'HR': '#999999'}
HP = {'PC':  dict(alpha=0.01, beta=0.01, eta_a=0.005, tau=None),
      'COV': dict(alpha=0.01, beta=0.01, eta_a=0.005, tau=1e-3),
      'HR':  dict(alpha=0.01, beta=0.01, eta_a=0.005, tau=None)}
PC = RULE_COL['PC']
RATIOS = [0.8, 0.2, 0.65, 0.35, 0.5]
SHADE  = {0.8:'#CFE3D4', 0.2:'#D6D9EC', 0.65:'#E7E0EC', 0.35:'#EAE3D0', 0.5:'#EFEFEF'}

# short-body / long-body PC settings (beta knob, noise fixed)
BODY = {'short body': 0.02, 'long body': 0.15}
SWEEP_BETAS = [0.02, 0.03, 0.04, 0.05, 0.1, 0.15]

# F3 colour scheme: purple -> rose -> amber across reward ratio
#   anchors = F3.ipynb CONDITIONS  (0.9,0.1)->#7A3E79  (0.5,0.5)->#B54156  (0.1,0.9)->#EFA831
RR_CMAP = mcolors.LinearSegmentedColormap.from_list('pm_rr', ['#7A3E79', '#B54156', '#EFA831'])
def rr_color(rr):                       # reward ratio in [0.1, 0.9] -> base colour
    return np.array(mcolors.to_rgb(RR_CMAP(np.clip((rr - 0.1) / 0.8, 0, 1))))

sig   = lambda z: 1/(1+np.exp(-z))
def logit(p): p = np.clip(p, 1e-3, 1-1e-3); return np.log(p/(1-p))
def rolling(x, w): k = np.ones(w)/w; return np.convolve(x, k, mode='same')

def nature_ax(ax, xlabel='', ylabel='', nticks=4, nx=None, ny=None):
    for sp in ('top','right'): ax.spines[sp].set_visible(False)
    for sp in ('left','bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx or nticks))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny or nticks))
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

# ── core simulator (baited two-armed bandit; arm A = index 0) ────────────────
def _stepper(rule='PC', n_seeds=16, beta=None, seed=0):
    """Jitted single-block scan + initial carry. `beta` overrides the rule's
    weight-update rate (used for the body-length sweep)."""
    h = HP[rule]
    hps = make_default_hps(seed)
    hps['alpha'] = jnp.array([h['alpha']])
    hps['beta'] = jnp.array([h['beta'] if beta is None else beta])
    hps['eta_a'] = jnp.array([h['eta_a']])
    if h['tau'] is not None: hps['tau'] = jnp.array([h['tau']])
    hb = batchify_hps(hps, n_seeds)
    uw, us = rules.rule2fns(rule)
    UA = jax.vmap(ssb.update_acts); UW = jax.vmap(uw); US = jax.vmap(us)
    NO = jax.vmap(ssb.act_noise);   EX = jax.vmap(ssb.expand_stimulus)
    env = jax.vmap(bandit_bait, in_axes=(0, 0, 0, 0))
    acts, states, weights, tern, key = init_params_batch(hb)
    def block(carry, probs_b, length):
        def step(c, _):
            acts, states, weights, bait, key = c
            r, act, bait, key = env(bait, acts[:, -2:], probs_b, key)
            stim = EX(r, acts)
            acts = UA(stim, acts, states, weights, tern, hb)
            acts, key = NO(acts, key, hb); acts = jnp.clip(acts, 0., 3.)
            weights = UW(stim, acts, states, weights, tern, hb)
            states = US(acts, states, hb)
            return (acts, states, weights, bait, key), (
                act, r[:, 0], acts[:, -2]-acts[:, -1], weights[:, 0, -2]-weights[:, 0, -1], acts[:, 0])
        return jax.lax.scan(step, carry, None, length=length)
    block = jax.jit(block, static_argnums=(2,))
    carry0 = (acts, states, weights, jnp.zeros((n_seeds, 2)), key)
    return block, carry0

def equilibrium(rule='PC', pA=0.225, pB=0.225, beta=None, n_seeds=16, burn=9000, sample=20000, seed=0):
    """Burn to equilibrium, then sample. Returns (act, dx, dw, rew, x0), each (T, n_seeds):
      act chosen-arm index; dx = xA-xB (output activity diff); dw = wA-wB (weight diff);
      rew reward signal; x0 input-neuron activity (shared trunk)."""
    block, carry = _stepper(rule, n_seeds, beta, seed)
    probs = jnp.tile(jnp.array([pA, pB])[None], (n_seeds, 1))
    carry, _ = block(carry, probs, burn)
    _, (act, rew, dx, dw, x0) = block(carry, probs, sample)
    return np.asarray(act), np.asarray(dx), np.asarray(dw), np.asarray(rew), np.asarray(x0)

def session(rule='PC', ratios=RATIOS, block_len=15000, n_seeds=8, beta=None,
            rng_seed=5, total=0.45, seed=0):
    """Continuous reversal-block session, state carried across blocks, rich arm
    randomised per block. Returns (action (T,S), bounds=[(start,end,rr),...])."""
    block, carry = _stepper(rule, n_seeds, beta, seed)
    rng = np.random.default_rng(rng_seed)
    order = []
    while len(order) < len(ratios)*2: order.extend(rng.permutation(len(ratios)).tolist())
    acts, bounds, t0 = [], [], 0
    for ci in order:
        rr = ratios[ci]
        if rng.random() < 0.5: rr = 1-rr
        probs = jnp.tile(jnp.array([total*rr, total*(1-rr)])[None], (n_seeds, 1))
        carry, (act, rew, dx, dw, _x0) = block(carry, probs, block_len)
        acts.append(np.asarray(act)); bounds.append((t0, t0+block_len, rr)); t0 += block_len
    return np.concatenate(acts), bounds

# ── metrics ──────────────────────────────────────────────────────────────────
def branchLW(dx, dw):
    """Attractor body length: occupancy-weighted L/W of the two choice branches
    (split by sign of dw), each as PC1/PC2 extent in the (dx, dw) plane."""
    out = []
    for sgn in (1, -1):
        m = np.sign(dw.ravel()) == sgn
        if m.sum() < 500: continue
        pts = np.column_stack([dx.ravel()[m], dw.ravel()[m]]); pts = pts - pts.mean(0)
        _, _, vt = np.linalg.svd(pts, full_matrices=False); pr = pts @ vt.T
        L = np.percentile(pr[:, 0], 99) - np.percentile(pr[:, 0], 1)
        W = np.percentile(pr[:, 1], 99) - np.percentile(pr[:, 1], 1)
        out.append((L/max(W, 1e-3), m.sum()))
    return np.average([o[0] for o in out], weights=[o[1] for o in out]) if out else np.nan

def windowed_choice_sd(act, n_windows=10):
    """Within-block choice variability: SD of the choice ratio across equal time
    windows of one equilibrium run (T,S). This is THE within-block SD metric used
    everywhere below (matching-curve error bars, panel annotation, and summary)."""
    T, S = act.shape; w = T // n_windows
    ch = (act[:w*n_windows] == 0).astype(float).reshape(n_windows, w, S).mean(1)
    return float(ch.std(0).mean())

def smooth_block(y, w):
    """Edge-safe centred moving average within one block (no cross-boundary smear,
    no end droop). Pure numpy; replaces np.convolve(mode='same')."""
    w = int(max(3, min(w, len(y))));  w += (w % 2 == 0)
    pad = w // 2
    return np.convolve(np.pad(y, pad, mode='edge'), np.ones(w)/w, mode='valid')

# ── body-length experiment ───────────────────────────────────────────────────
def cfg(full=False):
    if full:
        return dict(ratios=np.linspace(0.1, 0.9, 9), n_seeds=40, burn=40000, sample=40000, nwin=20)
    return dict(ratios=np.linspace(0.15, 0.85, 5), n_seeds=6, burn=3000, sample=4000, nwin=8)

def run_bodylen(full=False, total=0.45):
    """Per body & reward ratio: equilibrium choice fraction, per-ratio within-block
    choice SD, mean dw, and the (x0, dx, dw) cloud. L/W and the body's characteristic
    within-block SD are both measured on the SAME symmetric-reward run (so the body
    sits exactly on the summary sweep). Returns (nested dict, params)."""
    p = cfg(full); out = {}
    for label, om in BODY.items():
        rows, clouds = [], []
        for rr in p['ratios']:
            act, dx, dw, rew, x0 = equilibrium('PC', total*rr, total*(1-rr), beta=om,
                                               n_seeds=p['n_seeds'], burn=p['burn'], sample=p['sample'])
            rows.append((rr, (act == 0).mean(), windowed_choice_sd(act, p['nwin']), dw.mean()))
            clouds.append((rr, x0.ravel(), dx.ravel(), dw.ravel()))
        # symmetric-reward run: defines BOTH the body length and its characteristic SD
        a0, dx0, dw0, _, _x0s = equilibrium('PC', total*0.5, total*0.5, beta=om,
                                            n_seeds=p['n_seeds'], burn=p['burn'], sample=p['sample'])
        out[label] = dict(rows=np.array(rows), clouds=clouds,
                          lw=branchLW(dx0, dw0),
                          within_sd=windowed_choice_sd(a0, p['nwin']))
    return out, p

def run_sweep(full=False, total=0.45):
    """beta sweep for the summary: (beta, L/W, within-block SD) at symmetric reward
    — identical metric/conditions to run_bodylen's symmetric run, so the two bodies
    lie on this curve."""
    p = cfg(full); rows = []
    for om in SWEEP_BETAS:
        act, dx, dw, _, _x0s = equilibrium('PC', total*0.5, total*0.5, beta=om,
                                           n_seeds=p['n_seeds'], burn=p['burn'], sample=p['sample'])
        rows.append((om, branchLW(dx, dw), windowed_choice_sd(act, p['nwin'])))
    return np.array(rows)

# density helpers for the geometry panels (F3 style)
def _dens2d(x, y, bins=50, smooth=1.0):
    h, xe, ye = np.histogram2d(x, y, bins=bins); h = gaussian_filter(h, smooth)
    xi = np.clip(np.digitize(x, xe) - 1, 0, bins-1); yi = np.clip(np.digitize(y, ye) - 1, 0, bins-1)
    return h[xi, yi]

def _dens3d(pts, bins=40, smooth=1.2):
    z = (pts - pts.mean(0)) / (pts.std(0) + 1e-12)
    h, edges = np.histogramdd(z, bins=bins); h = gaussian_filter(h, smooth)
    idx = [np.clip(np.digitize(z[:, d], edges[d]) - 1, 0, bins-1) for d in range(3)]
    return h[idx[0], idx[1], idx[2]]

print('setup ready — base ssb only')

## Panel 1 — attractor body length → within-block variability (headline)

Within-PC $\beta$ sweep (noise fixed). **Left:** the attractor body in the **$(x_A-x_B,\ w_A-w_B)$** plane — the plane its length is measured in. Clouds are coloured by reward ratio; for a long body the per-block equilibria string out *along* the manifold (so it reads as long), while a short body stays a tight blob. **Middle:** the probability-matching curve (choice ratio vs reward ratio); error bars = per-ratio within-block choice SD. **Right:** within-block choice SD vs body length (L/W) across the $\beta$ sweep, with the two bodies circled — they lie exactly on the curve because the SD and L/W use the *same* symmetric-reward run as the panel annotations.

A rotatable 3D version (elev/azim sliders, F3_mag-style) is at the **end of the notebook**.

Set `FULL=True` for the publication run (`burn=sample=40000`, `n_seeds=40`, 9 ratios).

In [ ]:
FULL = True   # <-- set True for the publication-scale run (heavy: ~minutes; run it yourself)
data, p = run_bodylen(full=FULL); sweep = run_sweep(full=FULL)
print('params:', {k:(v if np.isscalar(v) else f'{len(v)} ratios') for k, v in p.items()})
for label, d in data.items():
    print(f'{label:11s} L/W={d["lw"]:.2f}  within-block SD={d["within_sd"]:.3f}')

In [ ]:
# ── Panel 1 main figure: geometry (wA-wB vs xA-xB) + matching curve + summary ──
labels = list(BODY.keys())

# shared axis ranges across both bodies, in the plane where body length is measured
alldx = np.concatenate([c[2] for d in data.values() for c in d['clouds']])
alldw = np.concatenate([c[3] for d in data.values() for c in d['clouds']])
DXLIM = tuple(np.percentile(alldx, [0.2, 99.8]))
DWLIM = tuple(np.percentile(alldw, [0.2, 99.8]))

fig = plt.figure(figsize=(6.5, 2.7))
gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.25, 1.2], wspace=0.55, hspace=0.55)
rng = np.random.default_rng(0)

for ri, label in enumerate(labels):
    d = data[label]; rr_all = d['rows'][:, 0]

    # --- attractor body in (wA-wB, xA-xB): equilibria slide along it with reward ratio ---
    axg = fig.add_subplot(gs[ri, 0])
    for (rr, x0, dx, dw) in d['clouds']:                 # flat colour per reward ratio (match 3D cloud)
        idx = rng.choice(len(dx), min(3000, len(dx)), replace=False)
        X, Y = dw[idx], dx[idx]                       # x = wA-wB, y = xA-xB
        axg.scatter(X, Y, s=2, c=[rr_color(rr)],
                    alpha=0.22, linewidths=0, rasterized=True)
    axg.set_xlim(*DWLIM); axg.set_ylim(*DXLIM)
    nature_ax(axg, '$w_A-w_B$' if ri == 1 else '', '$x_A-x_B$', nx=3, ny=3)
    axg.set_title(f'{label}   L/W={d["lw"]:.1f}', fontsize=6.5, pad=2)

    # --- probability-matching curve (shaded error = per-ratio within-block choice SD) ---
    axm = fig.add_subplot(gs[ri, 1])
    ch, sd = d['rows'][:, 1], d['rows'][:, 2]
    sort_idx = np.argsort(rr_all)
    rr_sorted = rr_all[sort_idx]; ch_sorted = ch[sort_idx]; sd_sorted = sd[sort_idx]
    axm.plot([0, 1], [0, 1], 'k--', lw=0.6, zorder=1)
    axm.fill_between(rr_sorted, ch_sorted - sd_sorted, ch_sorted + sd_sorted,
                     color='#dddddd', alpha=0.6, zorder=2, edgecolor='none')
    axm.plot(rr_sorted, ch_sorted, color='#888888', lw=0.8, zorder=2.5)
    axm.scatter(rr_all, ch, s=16, c=[rr_color(r) for r in rr_all], zorder=3, edgecolors='w', linewidths=0.3)
    axm.set_xlim(0, 1); axm.set_ylim(0, 1)
    nature_ax(axm, 'reward ratio' if ri == 1 else '', 'choice ratio', nx=3, ny=3)
    axm.text(0.04, 0.96, f'within-block SD\n={d["within_sd"]:.2f}',
             transform=axm.transAxes, fontsize=6, va='top')

# --- summary: within-block SD vs L/W across the beta sweep (same metric as panels) ---
axs = fig.add_subplot(gs[:, 2])
axs.plot(sweep[:, 1], sweep[:, 2], '-', color='#999', lw=1, zorder=1)
sc = axs.scatter(sweep[:, 1], sweep[:, 2], c=np.log10(sweep[:, 0]),
                 cmap='Blues', s=18, zorder=2, edgecolors='w', linewidths=0.3)
for label in labels:                      # circle the two bodies — they sit on the curve
    d = data[label]
    axs.scatter([d['lw']], [d['within_sd']], s=80, facecolors='none',
                edgecolors=PC, linewidths=1.3, zorder=4)
    axs.annotate(label.split()[0], (d['lw'], d['within_sd']), textcoords='offset points',
                 xytext=(6, -1), fontsize=6, color=PC)
nature_ax(axs, 'attractor body length (L/W)', 'within-block choice SD', nx=4, ny=4)
cb = fig.colorbar(sc, ax=axs, fraction=0.05, pad=0.04)
cb.set_label('log$_{10}\\,\\beta$', fontsize=7); cb.ax.tick_params(labelsize=6)
axs.set_title('longer body → more variability', fontsize=7, pad=3)

# reward-ratio colourbar (shared by geometry + matching markers)
sm = plt.cm.ScalarMappable(cmap=RR_CMAP, norm=plt.Normalize(0.1, 0.9)); sm.set_array([])
cax = fig.add_axes([0.045, -0.04, 0.16, 0.025])
cb2 = fig.colorbar(sm, cax=cax, orientation='horizontal')
cb2.set_label('reward ratio (block)', fontsize=6); cb2.ax.tick_params(labelsize=6); cb2.set_ticks([0.1, 0.5, 0.9])

plt.savefig('f3_pm_variability.svg', dpi=300, bbox_inches='tight'); plt.show()
